# Ch 20. 教師あり微調整 — 解答

> このノートブックは 5 問すべての再現可能な解答を提供する。出力は消去済みで、コードセルは CPU のみの環境で上から順に実行できる。


## 問題 1

**問題**: 損失マスキングの有無で学習し、応答品質を比較せよ。

### 解答

応答のみのマスキングではプロンプト token の label を $-100$ として損失から除外する。同じ初期モデルと応答 token 予算で応答 NLL と固定評価 rubric を比較する。以下で応答位置だけが平均に含まれることを確認する。


In [ ]:
import numpy as np

# Train one-token response classifiers with and without prompt-loss masking.
rng = np.random.default_rng(2001)
samples, features = 400, 6
X = rng.normal(size=(samples,features)); response = (X[:,0]+X[:,1]>0).astype(int); prompt = rng.integers(0,2,size=samples)
results={}
for masked in (False, True):
    w=np.zeros(features); lr=0.4
    for _ in range(100):
        logits=X@w; prob=1/(1+np.exp(-logits))
        target=response if masked else (response+prompt)/2
        w -= lr * X.T@(prob-target)/samples
    response_nll=-np.mean(response*np.log(prob+1e-9)+(1-response)*np.log(1-prob+1e-9))
    results[masked]={"response_nll":float(response_nll),"accuracy":float(np.mean((prob>.5)==response))}
assert results[True]["response_nll"] < results[False]["response_nll"] and results[True]["accuracy"] > results[False]["accuracy"]
print({("masked" if k else "unmasked"): {m:round(v,4) for m,v in r.items()} for k,r in results.items()})


## 問題 2

**問題**: 学習率 $10^{-2},10^{-3},10^{-4}$ で SFT を行い比較せよ。

### 解答

大きな SFT 学習率は事前学習表現を急激に損なうことがある。同じステップで train/validation 応答損失、勾配ノルム、事前学習検証損失の変化を追い、過学習と catastrophic forgetting を区別する。


In [ ]:
import numpy as np

rng=np.random.default_rng(2002); X=rng.normal(size=(300,5)); true_w=rng.normal(size=5); y=X@true_w
pretrained=true_w+rng.normal(scale=.2,size=5); results={}
for lr in (1e-2,1e-3,1e-4):
    w=pretrained.copy(); curve=[]
    for _ in range(80):
        error=X@w-y; curve.append(float(np.mean(error**2))); w-=lr*2*X.T@error/len(X)
    results[lr]={"validation_mse":curve[-1],"distance_from_pretrained":float(np.linalg.norm(w-pretrained))}
assert results[1e-2]["validation_mse"] < results[1e-3]["validation_mse"] < results[1e-4]["validation_mse"]
print({str(k): {m:round(v,6) for m,v in r.items()} for k,r in results.items()})


## 問題 3

**問題**: instruction データを 5、50、500 件に増やしながら SFT の効果を比較せよ。

### 解答

データ数だけを変え更新 token 予算を固定すると多様性効果を分離できる。5 件では反復過学習、500 件では広い一般化が予想されるが、これは仮説であり固定した未学習指示集合で検証する。


In [ ]:
import numpy as np

# More diverse instructions cover more latent directions; evaluate on a held-out fixed set.
rng=np.random.default_rng(2003); pool=rng.normal(size=(500,12)); true_w=rng.normal(size=12); targets=pool@true_w
test=rng.normal(size=(300,12)); test_y=test@true_w; results={}
for n in (5,50,500):
    w,*_=np.linalg.lstsq(pool[:n],targets[:n],rcond=None)
    results[n]=float(np.mean((test@w-test_y)**2))
assert results[500] < 1e-20 and results[50] < results[5]
print({n:{"heldout_mse":f"{mse:.3e}"} for n,mse in results.items()})


## 問題 4

**問題**: SFT データを ChatML 形式に変換せよ。

### 解答

ChatML は各 message を role token、内容、終了 token で直列化する。tokenizer の実際の特殊 token 定義を使い、文字列を推測しない。以下の変換で構造保持と決定性を確認する。


In [ ]:
messages=[{"role":"system","content":"Be concise."},{"role":"user","content":"Add 2 and 3."},{"role":"assistant","content":"5"}]
def to_chatml(items):
    return "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in items)
serialized=to_chatml(messages)
for message in messages:
    assert f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>" in serialized
assert serialized == to_chatml(messages)
print(serialized)


## 問題 5

**問題**: 事前学習モデルと SFT モデルの応答を比較し、違いを説明せよ。

### 解答

事前学習モデルは次 token 分布を学ぶが、指示–応答境界に従うよう直接最適化されていない。SFT は応答形式と停止条件を強化する。事実性向上は自動保証されないため、blind rubric で有用性・正確性・形式を別々に評価する。


In [ ]:
import numpy as np

# Blind rubric on held-out toy instructions: format compliance and task correctness are independent checks.
instructions=[("add",2,3),("multiply",3,4),("add",5,8),("multiply",2,7)]
def pretrained_response(item): return str(item[1])
def sft_response(item): return str(item[1]+item[2] if item[0]=="add" else item[1]*item[2])
expected=["5","12","13","14"]
scores={}
for name,model in (("pretrained",pretrained_response),("sft",sft_response)):
    responses=[model(x) for x in instructions]
    scores[name]={"exact_accuracy":float(np.mean(np.array(responses)==expected)),"numeric_format":float(np.mean([r.isdigit() for r in responses]))}
assert scores["sft"]["exact_accuracy"] > scores["pretrained"]["exact_accuracy"]
print(scores)
